# Bootstrap & Resampling
**Topic:** Probability & Statistics

In [1]:
import numpy as np
import pandas as pd

import plotly.graph_objects as go

import ipywidgets as widgets
from ipywidgets import IntSlider, Output, VBox

from IPython.display import display, clear_output
from scipy import stats

from tkh_utils import PALETTE, FONT, base_layout
from tkh_utils import make_output, make_int_slider, display_widget, register_observer

---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** what a bootstrap sample is and how it differs from the original sample
- **Explain** why repeating a statistic across many bootstrap samples produces a reliable confidence interval
- **Interpret** a bootstrap distribution and compare it to a parametric confidence interval

> **Tip:** Start with a small number of bootstrap samples (around 50) and watch the distribution look ragged and unreliable. Increase to 2000 and see it smooth out into a stable bell shape.

---
## How we got here

In *14: Sampling Distributions* we learned that taking many samples from a population and computing a statistic each time produces a sampling distribution. In *16: Confidence Intervals* we built confidence intervals using formulas that assume the statistic follows a normal distribution. But what if the formula does not exist, or the normality assumption does not hold? The bootstrap sidesteps both problems by simulating the sampling process using only the data you already have.

---
## Why this matters for data science

The bootstrap is the statistical foundation behind cross-validation. When you split your data into folds and measure model performance on each, you are estimating how much your metric would vary across different samples from the same population, which is the core idea of resampling. Understanding bootstrap also helps you build honest confidence intervals around any model metric (accuracy, AUC, RMSE) rather than reporting a single number that implies false precision.

---
## Try it yourself

In [2]:
np.random.seed(42)
SAMPLE = np.random.lognormal(mean=3.8, sigma=0.5, size=80)
TRUE_MEAN = np.mean(SAMPLE)

# Parametric CI (t-based) for reference
se_param = stats.sem(SAMPLE)
t_crit   = stats.t.ppf(0.975, df=len(SAMPLE) - 1)
CI_PARAM = (TRUE_MEAN - t_crit * se_param, TRUE_MEAN + t_crit * se_param)

out = make_output()
n_slider = make_int_slider(50, 3000, 50, 50, "Bootstrap samples:")


def render(n_boot):
    with out:
        clear_output(wait=True)

        rng = np.random.default_rng(seed=0)
        boot_means = [
            np.mean(rng.choice(SAMPLE, size=len(SAMPLE), replace=True))
            for _ in range(n_boot)
        ]
        boot_means = np.array(boot_means)

        ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])
        ci_width_boot  = ci_hi - ci_lo
        ci_width_param = CI_PARAM[1] - CI_PARAM[0]

        fig = go.Figure()
        fig.add_trace(go.Histogram(
            x=boot_means,
            nbinsx=40,
            marker_color=PALETTE["primary"],
            opacity=0.65,
            name=f"Bootstrap means (n={n_boot})",
        ))
        fig.add_vline(x=TRUE_MEAN, line_width=2.5,
                      line_color=PALETTE["secondary"],
                      annotation_text=f"Sample mean: {TRUE_MEAN:.1f}",
                      annotation_position="top right")
        fig.add_vrect(x0=ci_lo, x1=ci_hi,
                      fillcolor=PALETTE["primary"], opacity=0.10,
                      line_width=0,
                      annotation_text=f"Bootstrap 95% CI: [{ci_lo:.1f}, {ci_hi:.1f}]  width={ci_width_boot:.1f}",
                      annotation_position="top left")
        fig.add_vrect(x0=CI_PARAM[0], x1=CI_PARAM[1],
                      fillcolor=PALETTE["accent"], opacity=0.12,
                      line_color=PALETTE["accent"], line_width=1.5,
                      annotation_text=f"Parametric 95% CI: [{CI_PARAM[0]:.1f}, {CI_PARAM[1]:.1f}]  width={ci_width_param:.1f}",
                      annotation_position="bottom right")

        layout = base_layout(
            title=f"Bootstrap Distribution of the Sample Mean  ({n_boot} resamples)",
            xaxis_title="Bootstrap sample mean",
            yaxis_title="Count",
        )
        layout.update(showlegend=True)
        fig.update_layout(layout)
        display(go.FigureWidget(fig))


register_observer([n_slider], lambda change: render(n_slider.value))

display_widget(VBox([n_slider]), out)
render(n_slider.value)

---
## What's happening?

**A bootstrap sample** is created by drawing $n$ observations from your existing data *with replacement*. Some observations appear twice, some not at all — on average each bootstrap sample contains about 63.2% of the unique original observations. This mimics what would happen if you could go back and collect a new sample from the same population.

The algorithm:

1. Start with a sample of size $n$
2. Draw $n$ observations with replacement — this is one bootstrap sample
3. Compute your statistic of interest (mean, median, AUC, anything)
4. Repeat steps 2–3 many times (usually 1000–5000)
5. The distribution of those statistics is the **bootstrap distribution**
6. The 2.5th and 97.5th percentiles give a 95% confidence interval

### Bootstrap CI vs parametric CI

| Property | Parametric CI | Bootstrap CI |
|----------|--------------|-------------|
| Requires normality assumption | Yes (or CLT to kick in) | No |
| Works for any statistic | Only if formula exists | Yes — median, correlation, AUC, anything |
| Computationally cheap | Yes | Moderate (1000–5000 resamples) |
| Interval interpretation | Identical (frequentist coverage) | Identical |

Notice in the widget that both CIs are very similar when the sample size is large enough for the CLT to apply. The bootstrap earns its keep when the statistic is complex or the sample is small.

---
## Real-world example: Confidence around a model's accuracy

A classifier achieves 84% accuracy on a 150-sample test set. The single number tells you nothing about how stable that estimate is. The bootstrap gives you a distribution.

The chart below simulates test-set predictions from a 84%-accurate classifier, then bootstraps the accuracy 2000 times. Notice:

- The bootstrap distribution reveals that "84% accurate" really means "somewhere between 77% and 90% accurate"
- Reporting the CI is more honest than reporting a single point estimate
- This is exactly the logic behind repeated k-fold cross-validation: each fold is one resample

> **Discussion question:** If two models have overlapping bootstrap CIs for accuracy, what does that tell you about choosing between them?

In [3]:
np.random.seed(99)
n_test   = 150
true_acc = 0.84

# Simulate binary correct/wrong predictions
correct = np.random.binomial(1, true_acc, size=n_test)
point_acc = np.mean(correct)

rng = np.random.default_rng(seed=1)
boot_accs = [
    np.mean(rng.choice(correct, size=n_test, replace=True))
    for _ in range(2000)
]
boot_accs = np.array(boot_accs)
ci_lo, ci_hi = np.percentile(boot_accs, [2.5, 97.5])

fig = go.Figure()
fig.add_trace(go.Histogram(
    x=boot_accs * 100,
    nbinsx=30,
    marker_color=PALETTE["primary"],
    opacity=0.65,
    name="Bootstrap accuracy distribution",
))
fig.add_vline(x=point_acc * 100, line_color=PALETTE["secondary"], line_width=2.5,
              annotation_text=f"Point estimate: {point_acc:.1%}",
              annotation_position="top right")
fig.add_vrect(x0=ci_lo * 100, x1=ci_hi * 100,
              fillcolor=PALETTE["primary"], opacity=0.12, line_width=0,
              annotation_text=f"95% CI: [{ci_lo:.1%}, {ci_hi:.1%}]",
              annotation_position="top left")

layout = base_layout(
    title="Bootstrap Distribution of Test-Set Accuracy (n=150, 2000 resamples)",
    xaxis_title="Accuracy (%)",
    yaxis_title="Count",
)
layout.update(showlegend=True)
fig.update_layout(layout)
fig.show()

### Where resampling shows up in machine learning

| Technique | How it uses resampling |
|-----------|------------------------|
| k-fold cross-validation | Each fold is a different train/test split — estimates generalization variance |
| Bootstrap aggregating (bagging) | Each tree in a random forest trains on a bootstrap sample |
| Permutation feature importance | Shuffles one feature column — equivalent to resampling its relationship with the target |
| Bootstrap CI for any metric | Resample test predictions to get uncertainty bands around accuracy, AUC, RMSE |
| Out-of-bag evaluation | The ~36.8% of samples excluded from each bootstrap sample serve as a validation set |

---
## Key takeaway

> **The bootstrap turns your single sample into a simulation of many samples, letting you estimate uncertainty around any statistic — no formula required — which is the same logic that makes cross-validation a reliable measure of model performance.**

---
*Next up: Information Theory — how entropy and information gain explain what decision trees are actually doing when they split*